# NOVA Wolf Direct Worker — FREE / NO WORKFLOW

Прямой Colab-путь для команды **«Нова, включи Colab»**.

**Не зависит от GitHub Actions workflow.** Notebook читает файлы прямо из `main`, поднимает Blender worker и HTTPS tunnel.

Поток: **NOVA → Colab Free → Blender 5.2.1 LTS → 10s / 24fps / TRUE 360° → MP4 → NOVA → runtime stop**.

Если Google требует `Connect`, выбор T4 или `Run all`, это единственное ручное действие. Платные API и WanGP здесь не используются.


In [ ]:
# 1) FREE GPU gate
import os, secrets, subprocess, sys, time, re, json, threading, urllib.request
from pathlib import Path

PORT = 7861
TOKEN = secrets.token_urlsafe(24)
STOP_MARKER = Path('/content/NOVA_STOP_RUNTIME')
STOP_MARKER.unlink(missing_ok=True)

probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(probe.stdout if probe.stdout else 'GPU пока не обнаружен.')
if probe.returncode != 0:
    raise RuntimeError('Google не выдал GPU. Runtime → Change runtime type → T4 GPU, затем Run all снова.')
print('NOVA DIRECT: FREE GPU detected')


In [ ]:
# 2) Blender 5.2.1 LTS + FFmpeg + NOVA files
env = os.environ.copy(); env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo','apt-get','update','-qq'], check=True, env=env)
subprocess.run(['sudo','apt-get','install','-y','--no-install-recommends','ffmpeg','python3-venv','wget','xz-utils'], check=True, env=env)
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn','python-multipart','requests'], check=True)

BLENDER_DIR = Path('/content/blender-5.2.1-linux-x64')
BLENDER = BLENDER_DIR/'blender'
if not BLENDER.is_file():
    archive = Path('/content/blender-5.2.1-linux-x64.tar.xz')
    url = 'https://download.blender.org/release/Blender5.2/blender-5.2.1-linux-x64.tar.xz'
    urllib.request.urlretrieve(url, archive)
    subprocess.run(['tar','-xf',str(archive),'-C','/content'], check=True)
if not BLENDER.is_file():
    raise RuntimeError('Blender 5.2.1 install failed')
os.environ['NOVA_BLENDER_BIN'] = str(BLENDER)

REPO = Path('/content/nova-robot')
if (REPO/'.git').exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/magomedt149/nova-robot.git',str(REPO)], check=True)

required = [
    REPO/'automation/remote_gpu_worker.py',
    REPO/'automation/remote_gpu_worker_colab_auto.py',
    REPO/'blender-colab/scripts/render_wolf_cinema_auto.py',
]
for path in required:
    if not path.is_file():
        raise RuntimeError(f'Missing NOVA file: {path}')
subprocess.run([sys.executable,'-m','py_compile',*[str(p) for p in required]], check=True)
subprocess.run([str(BLENDER),'--version'], check=True)
subprocess.run(['ffmpeg','-version'], check=True, stdout=subprocess.DEVNULL)
print('NOVA DIRECT stack READY: Blender 5.2.1 + FFmpeg + worker')


In [ ]:
# 3) Protected worker + HTTPS tunnel
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)

worker_log = Path('/content/nova_wolf_worker.log')
tunnel_log = Path('/content/nova_wolf_tunnel.log')
os.environ['NOVA_REMOTE_TOKEN'] = TOKEN
os.environ['NOVA_REMOTE_JOB_ROOT'] = '/content/NOVA_REMOTE_JOBS'
os.environ['NOVA_RUNTIME_STOP_MARKER'] = str(STOP_MARKER)
os.environ['PATH'] = str(BLENDER_DIR) + os.pathsep + os.environ.get('PATH','')

worker_cmd = [sys.executable, str(REPO/'automation/remote_gpu_worker_colab_auto.py'), '--host','0.0.0.0','--port',str(PORT)]
worker = subprocess.Popen(worker_cmd, stdout=worker_log.open('w'), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(4)
if worker.poll() is not None:
    print(worker_log.read_text(errors='replace'))
    raise RuntimeError('NOVA direct wolf worker did not start')

tunnel = subprocess.Popen([str(cloudflared),'tunnel','--url',f'http://127.0.0.1:{PORT}','--no-autoupdate'],
                          stdout=tunnel_log.open('w'), stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    time.sleep(1)
    txt = tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0); break
if not url:
    print(tunnel_log.read_text(errors='replace'))
    raise RuntimeError('Cloudflare tunnel URL not found')

import requests
health = requests.get(url+'/health', headers={'X-NOVA-Token':TOKEN}, timeout=30).json()
if not health.get('gpu',{}).get('available') or not health.get('blender') or not health.get('ffmpeg'):
    print(json.dumps(health, ensure_ascii=False, indent=2))
    raise RuntimeError('Worker not ready: GPU / Blender / FFmpeg required')
print('NOVA DIRECT WOLF WORKER READY')
print(json.dumps(health, ensure_ascii=False, indent=2))


In [ ]:
# 4) Connect Code + return + automatic shutdown after NOVA safely copies MP4
connect_code = 'NOVA_CONNECT=' + json.dumps({'url':url,'token':TOKEN}, separators=(',',':'))
print('\n'+'='*78)
print('NOVA CONNECT CODE:', connect_code)
print('='*78)

from IPython.display import HTML, display
button_html = f'''
<button id="nova-copy" style="font-size:17px;padding:14px 18px;border-radius:12px;border:0;background:#111827;color:white;font-weight:800"
onclick="(async()=>{{try{{await navigator.clipboard.writeText({json.dumps(connect_code)})}}catch(e){{}};try{{window.opener&&window.opener.postMessage({json.dumps(connect_code)},'*')}}catch(e){{}};this.innerText='Connect Code ready ✓ Returning to NOVA…';setTimeout(()=>{{try{{window.close()}}catch(e){{}};setTimeout(()=>history.back(),500)}},550)}})()">COPY & RETURN TO NOVA</button>
'''
display(HTML(button_html))
print('Нажми COPY & RETURN TO NOVA. Дальше NOVA продолжит автоматически.')

def _runtime_stop_watchdog():
    while True:
        time.sleep(2)
        if STOP_MARKER.exists():
            print('NOVA: MP4 перенесён. Останавливаю Colab runtime…')
            try: tunnel.terminate()
            except Exception: pass
            try: worker.terminate()
            except Exception: pass
            time.sleep(1)
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception as exc:
                print('Automatic runtime stop was not accepted by Colab:', exc)
            return

threading.Thread(target=_runtime_stop_watchdog, daemon=True).start()
print('NOVA direct runtime-stop watcher: ACTIVE')
